# 04 Features and models (experiment log)
**Exploratory notebook.** The final feature code lives in `src/forecast_lib.py`; the code here is the first version of it, kept so the experiments can be reproduced.

**Compared on the same six walk-forward folds:** linear regression, random forest, LightGBM; with and without weather; raw vs residual vs ratio targets.

In [1]:
import pandas as pd, numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

h = pd.read_csv("../data/processed/hourly.csv", index_col=0, parse_dates=True)
y = h["total"]
hr, dow = y.index.hour, y.index.day_of_week

F = pd.DataFrame({"actual": y})
# calendar: always known in advance
F["hour_sin"] = np.sin(2 * np.pi * hr / 24)
F["hour_cos"] = np.cos(2 * np.pi * hr / 24)
F["is_weekend"] = (dow >= 5).astype(int)
F["dow"] = dow
# past-only demand (origin = midnight, so every lag must be >= 24 h)
F["lag_24"] = y.shift(24)
F["lag_168"] = y.shift(168)
F["daytype_lag"] = np.where(np.isin(dow, [1, 2, 3, 4]), F["lag_24"], F["lag_168"])
F["roll_mean_7d"] = y.shift(24).rolling(168).mean()
daily = y.resample("D").mean()
F["yday_mean"] = daily.shift(1).reindex(y.index.normalize()).to_numpy()
# weather: ACTUAL temperature, a stand-in for a forecast (optimistic!)
F["temp"] = h["Temperature"]
F["cdd20"] = np.maximum(F["temp"] - 20, 0)

CAL = ["hour_sin", "hour_cos", "is_weekend", "dow"]
LAGS = ["lag_24", "lag_168", "daytype_lag", "roll_mean_7d", "yday_mean"]
SET_A = CAL + LAGS                      # no weather
SET_B = SET_A + ["temp", "cdd20"]       # with weather

models = {
    "linear": lambda: make_pipeline(StandardScaler(), LinearRegression()),
    "rf": lambda: RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                        n_jobs=-1, random_state=0),
    "lgbm": lambda: lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05,
                                      random_state=0, verbose=-1),
}

tscv = TimeSeriesSplit(n_splits=6, test_size=24 * 28)   # same folds as notebook 03

def cv_mae(make_model, cols):
    out = []
    for tr, te in tscv.split(F):
        Xtr, ytr = F.iloc[tr][cols], F.iloc[tr]["actual"]
        ok = Xtr.notna().all(axis=1)                    # drop warm-up rows, train only
        m = make_model().fit(Xtr[ok], ytr[ok])
        out.append(mean_absolute_error(F.iloc[te]["actual"], m.predict(F.iloc[te][cols])))
    return out

res = {"baseline_daytype": [mean_absolute_error(F.iloc[te]["actual"], F.iloc[te]["daytype_lag"])
                            for _, te in tscv.split(F)]}
for name, mk in models.items():
    res[f"{name}_A_noweather"] = cv_mae(mk, SET_A)
    res[f"{name}_B_weather"] = cv_mae(mk, SET_B)

R = pd.DataFrame(res, index=[f"fold{i}" for i in range(1, 7)]).T
R["mean"] = R.mean(axis=1)
print(R.round(0).sort_values("mean"))

                     fold1   fold2   fold3   fold4   fold5   fold6    mean
linear_B_weather    2493.0  3150.0  1576.0  1193.0  1814.0  1376.0  1934.0
linear_A_noweather  2640.0  3148.0  1543.0  1183.0  1813.0  1383.0  1952.0
rf_A_noweather      2817.0  3767.0  1418.0  1063.0  1960.0  1260.0  2048.0
rf_B_weather        2885.0  3657.0  1477.0  1082.0  1973.0  1218.0  2049.0
lgbm_B_weather      3066.0  3431.0  1403.0  1117.0  1945.0  1510.0  2079.0
baseline_daytype    2307.0  4082.0  1608.0  1200.0  2175.0  1281.0  2109.0
lgbm_A_noweather    2779.0  4002.0  1452.0  1067.0  1938.0  1762.0  2167.0


In [2]:
def cv_predict(make_model, cols, residual=False):
    out = []
    for i, (tr, te) in enumerate(tscv.split(F), start=1):
        Xtr = F.iloc[tr][cols]
        ytr = F.iloc[tr]["actual"] - (F.iloc[tr]["daytype_lag"] if residual else 0)
        ok = Xtr.notna().all(axis=1)
        m = make_model().fit(Xtr[ok], ytr[ok])
        p = m.predict(F.iloc[te][cols])
        if residual:
            p = p + F.iloc[te]["daytype_lag"].to_numpy()
        out.append(pd.DataFrame({"fold": i, "pred": p}, index=F.index[te]))
    return pd.concat(out)

base = pd.concat([pd.DataFrame({"fold": i, "pred": F.iloc[te]["daytype_lag"].to_numpy()},
                               index=F.index[te])
                  for i, (_, te) in enumerate(tscv.split(F), start=1)])

runs = {
    "baseline":     base,
    "lin_B":        cv_predict(models["linear"], SET_B),
    "lin_B_resid":  cv_predict(models["linear"], SET_B, residual=True),
    "lgbm_B":       cv_predict(models["lgbm"],   SET_B),
    "lgbm_B_resid": cv_predict(models["lgbm"],   SET_B, residual=True),
}

def fold_stats(oof):
    d = F[["actual"]].join(oof, how="inner")
    d["err"] = d["pred"] - d["actual"]          # negative = under-prediction
    return d.groupby("fold")["err"].agg(MAE=lambda e: e.abs().mean(), bias="mean")

stats = {k: fold_stats(v) for k, v in runs.items()}
mae  = pd.DataFrame({k: s["MAE"]  for k, s in stats.items()})
bias = pd.DataFrame({k: s["bias"] for k, s in stats.items()})
mae_ex2 = mae.drop(index=2).mean()
mae.loc["mean"] = mae.mean()
mae.loc["mean_ex_fold2"] = mae_ex2
print("MAE\n", mae.round(0), "\n\nBias (pred - actual)\n", bias.round(0))

MAE
                baseline   lin_B  lin_B_resid  lgbm_B  lgbm_B_resid
fold                                                              
1                2307.0  2493.0       2493.0  3066.0        2439.0
2                4082.0  3150.0       3150.0  3431.0        4178.0
3                1608.0  1576.0       1576.0  1403.0        1530.0
4                1200.0  1193.0       1193.0  1117.0        1177.0
5                2175.0  1814.0       1814.0  1945.0        2222.0
6                1281.0  1376.0       1376.0  1510.0        1639.0
mean             2109.0  1934.0       1934.0  2079.0        2198.0
mean_ex_fold2    1714.0  1690.0       1690.0  1808.0        1801.0 

Bias (pred - actual)
       baseline   lin_B  lin_B_resid  lgbm_B  lgbm_B_resid
fold                                                     
1        -87.0  1565.0       1565.0   972.0        -455.0
2       2702.0  2404.0       2404.0  2489.0        3298.0
3         96.0   412.0        412.0   151.0         142.0
4        33

In [3]:
# 1) level-adjusted seasonal naive: last week's same hour x (yesterday / same weekday a week earlier)
F["wow_ratio"] = (daily.shift(1) / daily.shift(8)).reindex(y.index.normalize()).to_numpy()
F["lvl_adj"] = F["lag_168"] * F["wow_ratio"]

# 2) scale-free features and target (past-only: roll_mean_7d is already shifted by 24 h)
rm = F["roll_mean_7d"]
for c in ["lag_24", "lag_168", "yday_mean"]:
    F[c + "_rel"] = F[c] / rm
REL = CAL + ["lag_24_rel", "lag_168_rel", "yday_mean_rel", "temp", "cdd20"]

def cv_predict_rel(make_model, cols):
    out = []
    for i, (tr, te) in enumerate(tscv.split(F), start=1):
        Xtr = F.iloc[tr][cols]
        ytr = F.iloc[tr]["actual"] / F.iloc[tr]["roll_mean_7d"]
        ok = Xtr.notna().all(axis=1) & ytr.notna()
        m = make_model().fit(Xtr[ok], ytr[ok])
        p = m.predict(F.iloc[te][cols]) * F.iloc[te]["roll_mean_7d"].to_numpy()
        out.append(pd.DataFrame({"fold": i, "pred": p}, index=F.index[te]))
    return pd.concat(out)

def oof_from_col(col):
    return pd.concat([pd.DataFrame({"fold": i, "pred": F.iloc[te][col].to_numpy()}, index=F.index[te])
                      for i, (_, te) in enumerate(tscv.split(F), start=1)])

def summarize(runs):
    t = pd.DataFrame({k: fold_stats(v)["MAE"] for k, v in runs.items()})
    ex2 = t.drop(index=2).mean()
    t.loc["mean"] = t.mean()
    t.loc["mean_ex_fold2"] = ex2
    return t

runs2 = {
    "baseline": base,
    "lvl_adj":  oof_from_col("lvl_adj"),
    "lin_rel":  cv_predict_rel(models["linear"], REL),
    "lgbm_rel": cv_predict_rel(models["lgbm"], REL),
}
print(summarize(runs2).round(0))

               baseline  lvl_adj  lin_rel  lgbm_rel
fold                                               
1                2307.0   2342.0   2587.0    2660.0
2                4082.0   3004.0   3404.0    3017.0
3                1608.0   2192.0   1739.0    1689.0
4                1200.0   1216.0   1343.0    1062.0
5                2175.0   2462.0   1842.0    1885.0
6                1281.0   1554.0   1526.0    1003.0
mean             2109.0   2128.0   2073.0    1886.0
mean_ex_fold2    1714.0   1953.0   1807.0    1660.0


In [4]:
def daily_mae(oof):
    d = F[["actual"]].join(oof, how="inner")
    return (d["pred"] - d["actual"]).abs().resample("D").mean()

diff = (daily_mae(runs2["baseline"]) - daily_mae(runs2["lgbm_rel"])).dropna()   # >0 = model better

def boot_ci(s, block=7, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    blocks = [s.iloc[i:i+block].to_numpy() for i in range(0, len(s), block)]
    means = [np.concatenate([blocks[j] for j in rng.integers(0, len(blocks), len(blocks))]).mean()
             for _ in range(n)]
    return np.percentile(means, [2.5, 97.5]).round(0)

eid = (diff.index >= "2017-09-01") & (diff.index <= "2017-09-07")
print("all days   : mean gain", round(diff.mean()), "| 95% CI", boot_ci(diff))
print("excl. Eid  : mean gain", round(diff[~eid].mean()), "| 95% CI", boot_ci(diff[~eid]))

all days   : mean gain 223 | 95% CI [-64. 531.]
excl. Eid  : mean gain 60 | 95% CI [-123.  224.]


## Conclusion
- With actual temperature included, no model beat the day-type baseline in every fold. Best was LightGBM predicting load as a ratio of the past-week mean (mean MAE 1,886 vs 2,109), but it lost folds 1 and 3.
- Residual target: a wash (for linear regression it is mathematically identical to the plain model). Level-adjusted naive: won only the Eid fold, worse everywhere else.
- Block-bootstrap interval of the gain over the baseline: [-64, 531] over all days and [-123, 224] excluding Eid week, so both include zero.
- About 12 variants were tried on the same folds, so the best one is optimistically biased.
- Next: notebook 05 tests whether dropping weather helps.